# K-Nearest Neighbors:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Importing Dataset:

In [2]:
data = pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM']),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'),  ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

X_train = preprocessing.fit_transform(X_train)
X_test = preprocessing.transform(X_test)

In [6]:
param = {'n_neighbors' : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}

grid = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_ * 100:.2f}")

Best Parameters: {'n_neighbors': 9}
Best Score: 79.34


In [7]:
knn = KNeighborsRegressor(n_neighbors=9)

start_time = time.time()
knn.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.00s


## Results:

In [8]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [9]:
# Results on Training Data:

y_pred_train = knn.predict(X_train)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 84.36%

mean absolute error(Training Data): 16.38%

adjusted r2 Score: 84.32%


In [10]:
# Results on Testing Data:

y_pred_test = knn.predict(X_test)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 81.03%

mean absolute error (Testing Data): 19.12%

adjusted r2 Score: 80.97%


## Cross-Val-Score:

In [11]:
preprocessing = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), ['Weight', 'PPI', 'Clock Speed', 'RAM Speed', 'Refresh Rate', 'Graphics Memory', 'SSD', 'HDD', 'RAM']),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'),  ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

knn_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', KNeighborsRegressor(n_neighbors=9))
])

In [12]:
print(f"r2 Score: {np.mean(cross_val_score(estimator=knn_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 79.95%


In [13]:
print(f"mean absolute error: {-np.mean(cross_val_score(estimator=knn_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

mean absolute error: 18.80%


---

# Conclusion:

- *KNN Model* performs better than *Linear Regression Model* with **r2-Score = 79.95% & M.A.P.E = 18.80%.**